# Pipeline 5: Incident Early Warning

This notebook trains two intake-time risk models:
- Self-harm risk (`has_self_harm`)
- Runaway risk (`has_runaway`)

The workflow follows the repo's deployment pattern:
1. Build ETL frame from residents + incidents
2. Train and compare candidate classifiers (recall-prioritized)
3. Select best model per target
4. Save combined model bundle + metadata + metrics via `incident_early_warning.artifacts`

In [1]:
from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# Match pattern used in other notebooks: add cwd/parents and ml/ child candidates.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
for p in candidates:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

for p in candidates:
    ml_path = p / "ml"
    if ml_path.exists() and str(ml_path) not in sys.path:
        sys.path.insert(0, str(ml_path))

try:
    from ml.incident_early_warning.artifacts import save_metadata, save_metrics, save_model_bundle
    from ml.incident_early_warning.etl import build_training_frame
except ModuleNotFoundError:
    from incident_early_warning.artifacts import save_metadata, save_metrics, save_model_bundle
    from incident_early_warning.etl import build_training_frame

In [2]:
df = build_training_frame()

feature_cols = [
    c
    for c in df.columns
    if c not in {"resident_id", "has_self_harm", "has_runaway"}
]

X = df[feature_cols].copy()
y_selfharm = df["has_self_harm"].astype(int)
y_runaway = df["has_runaway"].astype(int)

X_train, X_test, y_sh_train, y_sh_test = train_test_split(
    X,
    y_selfharm,
    test_size=0.2,
    random_state=42,
    stratify=y_selfharm,
)
_, _, y_rw_train, y_rw_test = train_test_split(
    X,
    y_runaway,
    test_size=0.2,
    random_state=42,
    stratify=y_runaway,
)

X_train.shape, X_test.shape

((48, 14), (12, 14))

In [3]:
candidates = {
    "logreg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "tree": DecisionTreeClassifier(max_depth=4, random_state=42, class_weight="balanced"),
    "rf": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    "gb": GradientBoostingClassifier(random_state=42),
}


def evaluate_models(X_tr, y_tr, X_te, y_te):
    rows = []
    fitted = {}
    for name, model in candidates.items():
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.35).astype(int)
        rows.append(
            {
                "model": name,
                "recall": recall_score(y_te, pred, zero_division=0),
                "precision": precision_score(y_te, pred, zero_division=0),
                "f1": f1_score(y_te, pred, zero_division=0),
                "roc_auc": roc_auc_score(y_te, proba) if len(np.unique(y_te)) > 1 else 0.5,
            }
        )
        fitted[name] = model
    return pd.DataFrame(rows).sort_values(["recall", "roc_auc"], ascending=False), fitted

selfharm_eval, selfharm_models = evaluate_models(X_train, y_sh_train, X_test, y_sh_test)
runaway_eval, runaway_models = evaluate_models(X_train, y_rw_train, X_test, y_rw_test)

print("Self-harm model ranking")
display(selfharm_eval)
print("Runaway model ranking")
display(runaway_eval)

Self-harm model ranking


,model,recall,precision,f1,roc_auc
0,logreg,1.0,0.5,0.666667,0.900
2,rf,0.5,0.5,0.500000,0.950
3,gb,0.5,0.5,0.500000,0.825
1,tree,0.5,1.0,0.666667,0.750


Runaway model ranking


,model,recall,precision,f1,roc_auc
0,logreg,1.00,0.400000,0.571429,0.562500
2,rf,0.75,0.750000,0.750000,0.843750
1,tree,0.75,0.300000,0.428571,0.546875
3,gb,0.50,0.666667,0.571429,0.718750


In [4]:
best_selfharm_name = selfharm_eval.iloc[0]["model"]
best_runaway_name = runaway_eval.iloc[0]["model"]

best_selfharm_model = selfharm_models[best_selfharm_name]
best_runaway_model = runaway_models[best_runaway_name]

save_model_bundle(best_selfharm_model, best_runaway_model, feature_cols)
save_metadata(
    model_type="dual_classifier",
    feature_list=feature_cols,
    train_rows=len(X_train),
    test_rows=len(X_test),
    total_rows=len(X),
)
save_metrics(
    selfharm_metrics={
        "selected_model": best_selfharm_name,
        "recall": float(selfharm_eval.iloc[0]["recall"]),
        "precision": float(selfharm_eval.iloc[0]["precision"]),
        "f1": float(selfharm_eval.iloc[0]["f1"]),
        "roc_auc": float(selfharm_eval.iloc[0]["roc_auc"]),
    },
    runaway_metrics={
        "selected_model": best_runaway_name,
        "recall": float(runaway_eval.iloc[0]["recall"]),
        "precision": float(runaway_eval.iloc[0]["precision"]),
        "f1": float(runaway_eval.iloc[0]["f1"]),
        "roc_auc": float(runaway_eval.iloc[0]["roc_auc"]),
    },
)

best_selfharm_name, best_runaway_name

('logreg', 'logreg')